# Continuous sampling solvers

> CEM, iCEM, MPPI and predictive sampling over continuous multi-agent actions.

In [ ]:
#| default_exp planning.solver.sampling

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import time
from typing import Any

import numpy as np
import torch

from stable_marl.protocols import Costable
from stable_marl.planning.solver.base import SolverBase, gather_agents, prepare_init_action

In [ ]:
#| export
class GaussianSolverBase(SolverBase):
    """
    Base of the continuous sampling solvers: a Gaussian over the candidates
    `(n_envs, H, A, action_dim)`, warm-started from `init_action` (completed by the model's actor
    when it is `Actionable`). Returns `actions` `(n_envs, H, A, action_dim)`.
    """
    discrete = False

    def __init__(self, cost: Costable, batch_size: int | None = 1, num_samples: int = 300, var_scale: float = 1.0,
                 n_steps: int = 30, **kwargs):
        super().__init__(cost, batch_size=batch_size, num_samples=num_samples, **kwargs)
        self.var_scale, self.n_steps = var_scale, n_steps

    def _n_envs_of(self, info_dict: dict) -> int:
        return len(next(v for v in info_dict.values() if isinstance(v, (torch.Tensor, np.ndarray, list))))

    def init_action_distrib(self, n_envs: int, actions: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
        "Mean (the warm start, zero-padded) and standard deviation (`var_scale`), (n_envs, H, A, action_dim)."
        shape = (n_envs, self.horizon, self._num_agents, self.action_dim)
        mean = torch.zeros(shape, dtype=self.dtype, device=self.device)
        if actions is not None:
            t = min(actions.shape[1], self.horizon)
            mean[:, :t] = actions[:, :t].to(self.device, self.dtype)
        return mean, torch.full(shape, float(self.var_scale), dtype=self.dtype, device=self.device)

    def _init(self, info_dict: dict, init_action: torch.Tensor | None) -> tuple[int, torch.Tensor, torch.Tensor]:
        n_envs = self._n_envs_of(info_dict)
        init_action = prepare_init_action(self.cost, info_dict, init_action, self.horizon, n_envs, self._num_agents,
                                          self.action_dim, device=self.device)
        return (n_envs, *self.init_action_distrib(n_envs, init_action))

    def _noise(self, B: int) -> torch.Tensor:
        return torch.randn(B, self.num_samples, self.horizon, self._num_agents, self.action_dim,
                           generator=self.torch_gen, device=self.device, dtype=self.dtype)

In [ ]:
show_doc(GaussianSolverBase.init_action_distrib)

In [ ]:
#| export
class CEMSolver(GaussianSolverBase):
    "Cross-entropy method: the Gaussian is refit to the `topk` best candidates (per agent in 'per_agent' mode)."
    supports_callbacks = True

    def __init__(self, cost: Costable, topk: int = 30, **kwargs):
        super().__init__(cost, **kwargs)
        self.topk = topk

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs, mean, var = self._init(info_dict, init_action)
        final_costs = []
        self._callbacks_reset()
        for start, end in self._batches(n_envs):
            B, m, v, infos = end - start, mean[start:end], var[start:end], self._expand(info_dict, start, end)
            self._callbacks_start_batch()
            for step in range(self.n_steps):
                candidates = self._noise(B) * v.unsqueeze(1) + m.unsqueeze(1)
                candidates[:, 0] = m
                costs = self._costs(self.cost.get_cost(infos, candidates), B, self.num_samples)
                top_costs, top = torch.topk(costs, k=self.topk, dim=1, largest=False)
                elites = gather_agents(candidates, top)
                prev_m, prev_v = m, v
                m, v = elites.mean(dim=1), elites.std(dim=1, correction=0)
                self._callbacks_step(step=step, candidates=candidates, costs=self._user(costs),
                                     topk_vals=self._user(top_costs), topk_inds=self._user(top), topk_candidates=elites,
                                     mean=m, var=v, prev_mean=prev_m, prev_var=prev_v)
            mean[start:end], var[start:end] = m, v
            final_costs += self._cost_list(top_costs.mean(dim=1))
        self._log(start_time)
        return self._callbacks_output({'actions': mean.cpu(), 'costs': final_costs, 'mean': [mean.cpu()], 'var': [var.cpu()]})


class ICEMSolver(GaussianSolverBase):
    """
    Improved CEM: temporally correlated (colored, exponent `noise_beta`) noise, momentum `alpha`
    on the Gaussian, the `n_elite_keep` previous elites kept as candidates, candidates clipped to
    the action bounds. Returns the mean (`return_mean`) or the best elite.
    """
    supports_callbacks = True

    def __init__(self, cost: Costable, topk: int = 30, noise_beta: float = 2.0, alpha: float = 0.1,
                 n_elite_keep: int = 5, return_mean: bool = True, **kwargs):
        super().__init__(cost, **kwargs)
        self.topk, self.noise_beta, self.alpha = topk, noise_beta, alpha
        self.n_elite_keep, self.return_mean = n_elite_keep, return_mean

    def _colored_noise(self, B: int) -> torch.Tensor:
        H = self.horizon
        shape = (B, self.num_samples, self._num_agents, self.action_dim, H)
        white = torch.randn(shape, generator=self.torch_gen, device=self.device, dtype=self.dtype)
        if H > 1:
            freqs = torch.fft.rfftfreq(H, device=self.device).to(self.dtype)
            freqs[0] = 1.0
            scale = freqs.pow(-self.noise_beta / 2)
            scale[0] = scale[1]
            colored = torch.fft.irfft(torch.fft.rfft(white, dim=-1) * scale, n=H, dim=-1)
            white = colored / colored.std(dim=-1, keepdim=True).clamp(min=1e-8)
        return white.permute(0, 1, 4, 2, 3)          # (B, S, H, A, action_dim)

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs, mean, var = self._init(info_dict, init_action)
        final_costs = []
        bounded = hasattr(self, '_low')
        self._callbacks_reset()
        for start, end in self._batches(n_envs):
            B, m, v, infos = end - start, mean[start:end], var[start:end], self._expand(info_dict, start, end)
            prev_elites = None
            self._callbacks_start_batch()
            for step in range(self.n_steps):
                candidates = self._colored_noise(B) * v.unsqueeze(1) + m.unsqueeze(1)
                candidates[:, 0] = m
                if prev_elites is not None:
                    n = min(self.n_elite_keep, prev_elites.shape[1])
                    candidates[:, 1:1 + n] = prev_elites[:, :n]
                if bounded:
                    candidates = candidates.clamp(self._low, self._high)
                costs = self._costs(self.cost.get_cost(infos, candidates), B, self.num_samples)
                top_costs, top = torch.topk(costs, k=self.topk, dim=1, largest=False)
                prev_elites = elites = gather_agents(candidates, top)
                prev_m, prev_v = m, v
                m = self.alpha * m + (1 - self.alpha) * elites.mean(dim=1)
                v = self.alpha * v + (1 - self.alpha) * elites.std(dim=1, correction=0)
                self._callbacks_step(step=step, candidates=candidates, costs=self._user(costs),
                                     topk_vals=self._user(top_costs), topk_inds=self._user(top), topk_candidates=elites,
                                     mean=m, var=v, prev_mean=prev_m, prev_var=prev_v,
                                     action_low=getattr(self, '_low', None), action_high=getattr(self, '_high', None))
            mean[start:end] = m if self.return_mean else elites[:, 0]
            var[start:end] = v
            final_costs += self._cost_list(top_costs.mean(dim=1) if self.return_mean else top_costs[:, 0])
        self._log(start_time)
        return self._callbacks_output({'actions': mean.cpu(), 'costs': final_costs, 'mean': [mean.cpu()], 'var': [var.cpu()]})


class MPPISolver(GaussianSolverBase):
    """
    Model predictive path integral: the mean becomes the average of the candidates (the `topk`
    best, if `topk` < `num_samples`) weighted by `softmax(-cost / temperature)` (per agent in
    'per_agent' mode); the noise scale stays `var_scale`.
    """
    def __init__(self, cost: Costable, topk: int | None = 30, temperature: float = 0.5, **kwargs):
        super().__init__(cost, **kwargs)
        self.topk, self.temperature = topk, temperature

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs, mean, var = self._init(info_dict, init_action)
        final_costs = []
        for start, end in self._batches(n_envs):
            B, m, v, infos = end - start, mean[start:end], var[start:end], self._expand(info_dict, start, end)
            for _ in range(self.n_steps):
                candidates = m.unsqueeze(1) + self._noise(B) * v.unsqueeze(1)
                candidates[:, 0] = m
                costs = self._costs(self.cost.get_cost(infos, candidates), B, self.num_samples)
                if self.topk is not None and self.topk < self.num_samples:
                    costs, top = torch.topk(costs, k=self.topk, dim=1, largest=False)
                    candidates = gather_agents(candidates, top)
                weights = torch.softmax(-(costs - costs.min(dim=1, keepdim=True).values) / self.temperature, dim=1)
                m = torch.einsum('bka,bkhax->bhax', weights, candidates)
            mean[start:end] = m
            final_costs += self._cost_list(costs.mean(dim=1))
        self._log(start_time)
        return {'actions': mean.cpu(), 'costs': final_costs, 'mean': [mean.cpu()], 'var': [var.cpu()]}


class PredictiveSamplingSolver(GaussianSolverBase):
    """
    Predictive sampling: one round of Gaussian perturbations (scale `noise_scale`) of the nominal
    plan (the warm start); keeps the best candidate (per agent in 'per_agent' mode).
    """
    def __init__(self, cost: Costable, noise_scale: float = 1.0, **kwargs):
        super().__init__(cost, **kwargs)
        self.noise_scale = noise_scale

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs, nominal, _ = self._init(info_dict, init_action)
        final_costs = []
        for start, end in self._batches(n_envs):
            B, m, infos = end - start, nominal[start:end], self._expand(info_dict, start, end)
            candidates = m.unsqueeze(1) + self._noise(B) * self.noise_scale
            candidates[:, 0] = m
            costs = self._costs(self.cost.get_cost(infos, candidates), B, self.num_samples)
            best = costs.argmin(dim=1, keepdim=True)
            nominal[start:end] = gather_agents(candidates, best)[:, 0]
            final_costs += self._cost_list(costs.gather(1, best)[:, 0])
        self._log(start_time)
        return {'actions': nominal.cpu(), 'costs': final_costs}

### Tests

In [ ]:
from types import SimpleNamespace
import gymnasium as gym
from stable_marl.planning.solver.base import Solver, PLANNING_MODES

class QuadraticCost:
    "Squared distance of candidates (B, S, H, A, X) to a target plan (H, A, X)."
    def __init__(self, target, per_agent=False):
        self.target, self.per_agent = target, per_agent
    def get_cost(self, info_dict, candidates):
        cost = (candidates - self.target).pow(2).sum(dim=(2, 4))
        return cost if self.per_agent else cost.sum(-1)
    def criterion(self, info_dict, candidates):
        return self.get_cost(info_dict, candidates)

cfg = SimpleNamespace(horizon=3, action_block=2)
space = gym.spaces.Box(-1, 1, (2, 2))                       # 2 agents, 2-d actions
target = torch.tensor([0.5, -0.3, 0.1, 0.8]).repeat(3, 2, 1)  # (H, A, action_block * 2)
for mode in PLANNING_MODES:
    cost = QuadraticCost(target, per_agent=mode == 'per_agent')
    # plain CEM's variance collapses early in high dimension: the joint plan (24 numbers) needs a larger population
    for solver, tol in ((CEMSolver(cost, batch_size=2, num_samples=500, n_steps=15, topk=50, var_scale=0.5, mode=mode), 0.05),
                        (ICEMSolver(cost, batch_size=2, num_samples=200, n_steps=15, topk=20, var_scale=0.5, mode=mode), 0.05),
                        (MPPISolver(cost, batch_size=2, num_samples=300, n_steps=30, topk=None, temperature=0.5, var_scale=0.3, mode=mode), 0.1),
                        (PredictiveSamplingSolver(cost, batch_size=2, num_samples=2000, noise_scale=0.6, mode=mode), None)):
        solver.configure(action_space=space, n_envs=2, config=cfg)
        assert isinstance(solver, Solver) and solver.action_dim == 4
        out = solver.solve({'x': torch.zeros(2, 1)})
        assert out['actions'].shape == (2, 3, 2, 4)
        if tol is None:   # one round of sampling: only an improvement on the nominal (zero) plan
            zero_cost = cost.get_cost({}, torch.zeros(1, 1, 3, 2, 4))[0, 0]
            assert (torch.tensor(out['costs']) < zero_cost).all(), (mode, out['costs'])
        else:
            err = (out['actions'] - target).abs().max().item()
            assert err < tol, (type(solver).__name__, mode, err)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()